# E3 development pilot — clean versus self-predicted memory

This pilot uses only the frozen DAVIS-train model-selection holdout. It checks that filled-bank frame-level utility and attention measurements are operational before the preregistered multi-sequence c0–c4 run. It is not a hypothesis test.

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path
from kaggle_secrets import UserSecretsClient

TEMP = Path('/kaggle/temp')
REPO = TEMP / 'AdeSEG'
OUT = Path('/kaggle/working/latent_utility_e3_pilot')
OUT.mkdir(parents=True, exist_ok=True)
started = time.time()

def mark(stage, state='running', **details):
    payload = {'stage': stage, 'state': state, 'elapsed_seconds': time.time() - started, **details}
    (OUT/'status.json').write_text(json.dumps(payload, indent=2) + '\n')
    print('[status]', json.dumps(payload, sort_keys=True), flush=True)

def run(command, cwd=None, timeout=900):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, env=os.environ.copy(), check=True, timeout=timeout)

try:
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception as exc:
    key = None
    print(f'[wandb] Kaggle secret unavailable ({type(exc).__name__}); results will be synced after download', flush=True)
os.environ.update({'PYTHONUNBUFFERED': '1', 'GIT_TERMINAL_PROMPT': '0'})
if key:
    os.environ.update({'WANDB_API_KEY': key, 'WANDB_BASE_URL': 'https://forge.coreweave.com/api/wandb', 'WANDB_PROJECT': 'ade-seg', 'WANDB_ENTITY': 'adenoid-hypertrophy'})
mark('bootstrap')

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)
run(['git', 'clone', '--depth', '1', '--branch', 'experiment/utility-aware-memory-2026-10-16', 'https://github.com/aCoderChild/AdeSEG.git', REPO], timeout=300)
missing = [package for module, package in {'hydra': 'hydra-core>=1.3.2', 'iopath': 'iopath>=0.1.10', 'wandb': 'wandb'}.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, '-m', 'pip', 'install', *missing], timeout=300)
if importlib.util.find_spec('tensordict') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'tensordict>=0.7,<0.8'], timeout=300)
import torch
assert torch.cuda.is_available(), 'GPU required'
PROTOCOL = REPO/'protocols/latent_utility_v2.json'
MANIFEST = TEMP/'davis.jsonl'
run([sys.executable, REPO/'scripts/build_davis_manifest.py', '--data-root', '/kaggle/input', '--protocol', PROTOCOL, '--output', MANIFEST], cwd=REPO, timeout=180)
CKPT = TEMP/'sam2.1_hiera_base_plus.pt'
run(['wget', '--quiet', '--timeout=30', '--tries=3', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt', '-O', CKPT], timeout=600)
mark('ready', commit=subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip(), accelerator=torch.cuda.get_device_name(0))

In [ ]:
wandb_args = ['--wandb-base-url', os.environ['WANDB_BASE_URL']] if key else []
common = [sys.executable, REPO/'scripts/generate_utility_labels.py', '--sam2-cfg', 'configs/sam2.1_hiera_b+1024.yaml', '--sam2-checkpoint', CKPT, '--manifest', MANIFEST, '--split', 'holdout', '--video-ids', 'bus', '--label-ids', '1', '--protocol', PROTOCOL, '--device', 'cuda', '--max-analyzed-frames', '3', '--frame-selection', 'every_n', '--every-n', '15', '--minimum-target-frame', '15', *wandb_args]
conditions = {'self_predicted': ['--memory-source', 'predicted'], 'c0_clean': ['--memory-source', 'ground_truth']}
for name, extra in conditions.items():
    mark(name)
    condition_wandb = ['--wandb-name', f'latent-utility-e3-pilot-{name}'] if key else ['--wandb-mode', 'disabled']
    run([*common, '--output-dir', OUT/name, *condition_wandb, *extra], cwd=REPO, timeout=10800)

In [ ]:
import numpy as np

def rows(name):
    return [json.loads(line) for line in (OUT/name/'frames.jsonl').read_text().splitlines() if line]

result = {'scope': 'development pilot; not a hypothesis test', 'sequence_object': 'bus:1', 'conditions': {}}
for name in conditions:
    current = rows(name)
    result['conditions'][name] = {
        'frames': [row['frame'] for row in current],
        'mean_dice': float(np.mean([row['baseline_dice'] for row in current])),
        'mean_spearman': float(np.nanmean([row['attention_spearman'] for row in current])),
        'mean_harmful_attention_mass': float(np.nanmean([sum(a for a, u in zip(row['attention'], row['utility']) if u < 0) / sum(row['attention']) for row in current])),
        'mean_oracle_gain': float(np.mean([row['oracle_retrieval_dice'] - row['baseline_dice'] for row in current])),
        'memory_units': [len(row['units']) for row in current],
    }
result['clean_minus_self_spearman'] = result['conditions']['c0_clean']['mean_spearman'] - result['conditions']['self_predicted']['mean_spearman']
(OUT/'pilot_summary.json').write_text(json.dumps(result, indent=2) + '\n')
mark('complete', state='complete', summary=result)
print(json.dumps(result, indent=2), flush=True)